In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("OPENAI_API_KEY is set.")
else:
    print("OPENAI_API_KEY is not set.")

OPENAI_API_KEY is set.


In [13]:
from langchain_openai import ChatOpenAI

In [14]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [15]:
response = llm.invoke(input="What is the capital of France?")
response.content

'Paris'

# **RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA**

#### **Step-1: Preparing documents for your texts**

In [16]:
from langchain_core.documents import Document

## text data
my_text = """Once upon a time there was a kingdom far far away. The birds sang in the morning, and the villagers worked the fields under a golden sun. But deep in the Whispering Woods, a secret waited for a hero brave enough to find it. The secret was a magical amulet that could grant any wish, but it was guarded by a fierce dragon. Many had tried to retrieve the amulet, but none had succeeded. One day, a young adventurer named Aria decided to take on the challenge. With her trusty sword and a heart full of courage, she ventured into the Whispering Woods, ready to face whatever dangers lay ahead. The villagers watched as Aria disappeared into the dense forest, hoping for her safe return. As she journeyed deeper, the sounds of the village faded away, replaced by the eerie silence of the woods. Aria's heart pounded with both fear and excitement, knowing that the fate of the kingdom could rest on her success. Along the way, she encountered mystical creatures and overcame treacherous obstacles, each step bringing her closer to the dragon's lair and the coveted amulet."""

doc = Document(page_content=my_text, metadata={"source": "my_story.txt"})
doc


Document(metadata={'source': 'my_story.txt'}, page_content="Once upon a time there was a kingdom far far away. The birds sang in the morning, and the villagers worked the fields under a golden sun. But deep in the Whispering Woods, a secret waited for a hero brave enough to find it. The secret was a magical amulet that could grant any wish, but it was guarded by a fierce dragon. Many had tried to retrieve the amulet, but none had succeeded. One day, a young adventurer named Aria decided to take on the challenge. With her trusty sword and a heart full of courage, she ventured into the Whispering Woods, ready to face whatever dangers lay ahead. The villagers watched as Aria disappeared into the dense forest, hoping for her safe return. As she journeyed deeper, the sounds of the village faded away, replaced by the eerie silence of the woods. Aria's heart pounded with both fear and excitement, knowing that the fate of the kingdom could rest on her success. Along the way, she encountered my

#### **Step-2: Splitting the document in chunks.**

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents([doc])
chunks

[Document(metadata={'source': 'my_story.txt'}, page_content='Once upon a time there was a kingdom far far away. The birds sang in the morning, and the villagers worked the fields under a golden sun. But deep in the Whispering Woods, a secret waited for a hero brave enough to find it. The secret was a magical amulet that could grant any wish, but it was guarded by a fierce dragon. Many had tried to retrieve the amulet, but none had succeeded. One day, a young adventurer named Aria decided to take on the challenge. With her trusty sword and a heart full of'),
 Document(metadata={'source': 'my_story.txt'}, page_content="With her trusty sword and a heart full of courage, she ventured into the Whispering Woods, ready to face whatever dangers lay ahead. The villagers watched as Aria disappeared into the dense forest, hoping for her safe return. As she journeyed deeper, the sounds of the village faded away, replaced by the eerie silence of the woods. Aria's heart pounded with both fear and ex

#### **Step-3: Creating Embeddings for the chunks.**

In [18]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    check_embedding_ctx_length=False,
)

In [19]:
# convert this text to 1530 length vector
# embedding_model.embed_query("What is the capital of France?")

#### **Step-4: Create & Store Embeddings in Vector Store.**

In [20]:
from langchain_community.vectorstores import Chroma

vectorstore: Chroma = Chroma.from_documents(chunks, embedding_model)
vectorstore

#### **Step-5: Talk to LLM.**

In [21]:
context = vectorstore.similarity_search("What is AI?", k=1)
context

[Document(metadata={'source': 'my_story.txt'}, page_content="With her trusty sword and a heart full of courage, she ventured into the Whispering Woods, ready to face whatever dangers lay ahead. The villagers watched as Aria disappeared into the dense forest, hoping for her safe return. As she journeyed deeper, the sounds of the village faded away, replaced by the eerie silence of the woods. Aria's heart pounded with both fear and excitement, knowing that the fate of the kingdom could rest on her success. Along the way, she encountered mystical creatures")]

In [22]:
response = llm.invoke(f"Answer the question based on the following context: {context[0].page_content} \n\n Question: What is AI?")
response.content

'AI stands for Artificial Intelligence. It refers to computer systems or software that can perform tasks that normally require human intelligence, such as learning from data, reasoning, understanding language, recognizing patterns, and making decisions.\n\n- Types: Narrow (or weak) AI, which is designed for specific tasks, and General AI (hypothetical), which would match human-level versatility.\n- Common methods: machine learning, neural networks, deep learning.\n- Examples: voice assistants, image recognition, spam filters, recommendation systems.'